In [1]:
import os
import pandas as pd
import numpy as np
import logging
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, LSTM, GRU, MultiHeadAttention, LayerNormalization
from tensorflow.keras.layers import Dropout, BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.layers import multiply, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
import joblib
from typing import List, Tuple, Dict
import ta

class Config:
    # Parâmetros de dados
    MIN_VOLUME = 2000
    TOP_LEVELS = 10
    
    # Parâmetros do modelo
    SEQUENCE_LENGTH = 60
    BATCH_SIZE = 32
    EPOCHS = 100
    LEARNING_RATE = 1e-4
    
    # Parâmetros dos indicadores técnicos
    SMA_PERIOD = 20
    EMA_PERIOD = 20
    RSI_PERIOD = 9
    ADX_PERIOD = 9
    
    # Diretórios
    BASE_DIR = "C:/ARQUIVOS/OneDrive/Bolsa/COLETA/MT/FUTUROS/"
    CACHE_DIR = os.path.join(BASE_DIR, "cache")
    
    # Arquivos
    MODEL_FILE = "model.keras"
    SCALER_FILE = "scaler.pkl"
    REPORT_FILE = "market_report.txt"
    LOG_FILE = "market_analyzer.log"

class MarketReport:
    def __init__(self, config: Config):
        self.config = config
        self.report_template = """
{separator}
ANÁLISE DE MERCADO - {symbol}
{separator}
Data/Hora: {datetime}
Último Fechamento: {last_close:.2f}

{separator}
RESUMO DO MERCADO
{separator}
Tendência: {trend}
Volatilidade: {volatility:.2f}%
Volume Médio (5 períodos): {avg_volume:,.0f}
RSI Atual: {rsi:.1f}

{separator}
TOP NÍVEIS DE RESISTÊNCIA (Ordenados por Volume)
{separator}
{resistance_levels}

{separator}
TOP NÍVEIS DE SUPORTE (Ordenados por Volume)
{separator}
{support_levels}

{separator}
ANÁLISE ADICIONAL
{separator}
Gaps Importantes:
{gaps}

Zonas de Confluência:
{confluence_zones}

Próximos Níveis Críticos:
↑ Resistência: {next_resistance:.2f} (distância: {resistance_distance:.2f}%)
↓ Suporte: {next_support:.2f} (distância: {support_distance:.2f}%)

{separator}
ESTATÍSTICAS DE VOLUME
{separator}
Distribuição de Volume por Nível:
{volume_distribution}

{separator}
Gerado por Market Analyzer v2.0
Nota: Níveis são baseados em análise histórica e devem ser usados em conjunto com outras ferramentas.
{separator}
"""
        self.separator = "="*80

    def format_level_detail(self, level: dict, last_close: float) -> str:
        """Formata detalhes de um nível individual"""
        strength_bar = self._create_strength_bar(level['strength'])
        volume_bar = self._create_volume_bar(level['volume'])
        
        return (
            f"Preço: {level['price']:>10,.2f} | "
            f"Volume: {level['volume']:>10,.0f} {volume_bar} | "
            f"Força: {strength_bar} | "
            f"Toques: {level['touches']:>2d} | "
            f"Dist: {level['dist']:>6.2f}% | "
            f"Último Teste: {level['last_test']} | "
            f"Gap: {level['gap']*100:>4.1f}%"
        )

    def _create_strength_bar(self, strength: float, length: int = 10) -> str:
        """Cria uma barra visual representando a força do nível"""
        filled = int(strength * length)
        return f"[{'■' * filled}{'□' * (length - filled)}]"

    def _create_volume_bar(self, volume: float, max_volume: float = None, length: int = 5) -> str:
        """Cria uma barra visual representando o volume relativo"""
        if max_volume is None:
            max_volume = volume * 2
        filled = int((volume / max_volume) * length)
        return f"[{'▉' * filled}{'░' * (length - filled)}]"

    def format_levels(self, levels: List[dict], last_close: float) -> str:
        """Formata a lista de níveis para o relatório"""
        if not levels:
            return "Nenhum nível identificado"
            
        formatted_levels = []
        for i, level in enumerate(levels, 1):
            prefix = f"{i:2d}."
            details = self.format_level_detail(level, last_close)
            formatted_levels.append(f"{prefix} {details}")
            
            # Adiciona histórico de toques se houver mais de um
            if len(level['dates']) > 1:
                touch_history = "    ├─ Histórico: " + " → ".join(level['dates'][-3:])
                if len(level['dates']) > 3:
                    touch_history += f" (+ {len(level['dates'])-3} anteriores)"
                formatted_levels.append(touch_history)
                
            # Adiciona informações de confluência se existirem
            if 'confluence' in level and level['confluence']:
                confluence = "    └─ Confluência: " + ", ".join(level['confluence'])
                formatted_levels.append(confluence)
                
        return "\n".join(formatted_levels)

    def analyze_market_context(self, df: pd.DataFrame) -> dict:
        """Analisa o contexto geral do mercado"""
        last_price = df['ÚLT. PREÇO'].iloc[-1]
        sma20 = df['sma_20'].iloc[-1]
        
        # Determina tendência
        if last_price > sma20 * 1.02:
            trend = "ALTA ↑"
        elif last_price < sma20 * 0.98:
            trend = "BAIXA ↓"
        else:
            trend = "LATERAL →"
            
        # Calcula volatilidade
        volatility = df['atr'].iloc[-1] / last_price * 100
        
        # Volume médio
        avg_volume = df['VOL.'].tail(5).mean()
        
        return {
            'trend': trend,
            'volatility': volatility,
            'avg_volume': avg_volume,
            'rsi': df['rsi'].iloc[-1]
        }

    def find_gaps(self, df: pd.DataFrame) -> List[dict]:
        """Identifica gaps significativos"""
        gaps = []
        for i in range(1, len(df)):
            prev_high = df['PREÇO MÁX.'].iloc[i-1]
            curr_low = df['PREÇO MÍN.'].iloc[i]
            prev_low = df['PREÇO MÍN.'].iloc[i-1]
            curr_high = df['PREÇO MÁX.'].iloc[i]
            
            # Gap de alta
            if curr_low > prev_high:
                gaps.append({
                    'type': 'alta',
                    'size': curr_low - prev_high,
                    'date': df['DATA'].iloc[i],
                    'filled': curr_low <= df['PREÇO MÍN.'].iloc[i:].min()
                })
            # Gap de baixa
            elif curr_high < prev_low:
                gaps.append({
                    'type': 'baixa',
                    'size': prev_low - curr_high,
                    'date': df['DATA'].iloc[i],
                    'filled': curr_high >= df['PREÇO MÁX.'].iloc[i:].max()
                })
                
        return sorted(gaps, key=lambda x: x['size'], reverse=True)[:3]

    def generate_report(self, df: pd.DataFrame, supports: List[dict], 
                       resistances: List[dict], last_close: float) -> str:
        """Gera o relatório completo"""
        context = self.analyze_market_context(df)
        gaps = self.find_gaps(df)
        
        # Encontra próximos níveis críticos
        next_resistance = min((r['price'] for r in resistances if r['price'] > last_close), default=last_close*1.01)
        next_support = max((s['price'] for s in supports if s['price'] < last_close), default=last_close*0.99)
        
        # Formata a seção de gaps
        gaps_text = "\n".join(
            f"→ {gap['date'].strftime('%Y-%m-%d')}: Gap de {gap['type'].upper()} "
            f"({gap['size']:.2f} pontos) - {'Preenchido' if gap['filled'] else 'Aberto'}"
            for gap in gaps
        )
        
        # Calcula distribuição de volume
        volume_distribution = self._calculate_volume_distribution(df)
        
        return self.report_template.format(
            separator=self.separator,
            symbol="WDO",
            datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            last_close=last_close,
            trend=context['trend'],
            volatility=context['volatility'],
            avg_volume=context['avg_volume'],
            rsi=context['rsi'],
            resistance_levels=self.format_levels(resistances, last_close),
            support_levels=self.format_levels(supports, last_close),
            gaps=gaps_text,
            confluence_zones=self._format_confluence_zones(supports, resistances),
            next_resistance=next_resistance,
            next_support=next_support,
            resistance_distance=((next_resistance/last_close)-1)*100,
            support_distance=((last_close/next_support)-1)*100,
            volume_distribution=volume_distribution
        )
        
    def _calculate_volume_distribution(self, df: pd.DataFrame) -> str:
        """Calcula e formata a distribuição de volume por faixas de preço"""
        price_range = df['PREÇO MÁX.'].max() - df['PREÇO MÍN.'].min()
        num_bins = 5
        bin_size = price_range / num_bins
        
        volume_dist = []
        current_price = df['PREÇO MÍN.'].min()
        
        for i in range(num_bins):
            price_min = current_price
            price_max = current_price + bin_size
            mask = (df['PREÇO MÍN.'] >= price_min) & (df['PREÇO MÁX.'] < price_max)
            volume = df.loc[mask, 'VOL.'].sum()
            
            volume_dist.append({
                'range': f"{price_min:.2f} - {price_max:.2f}",
                'volume': volume,
                'percentage': volume / df['VOL.'].sum() * 100
            })
            
            current_price += bin_size
            
        # Formata a distribuição
        return "\n".join(
            f"→ {d['range']}: {d['volume']:,.0f} ({d['percentage']:.1f}%)"
            for d in sorted(volume_dist, key=lambda x: x['volume'], reverse=True)
        )
        
    def _format_confluence_zones(self, supports: List[dict], 
                               resistances: List[dict], threshold: float = 0.001) -> str:
        """Identifica e formata zonas de confluência entre níveis"""
        all_levels = [(s['price'], 'Suporte') for s in supports] + \
                    [(r['price'], 'Resistência') for r in resistances]
                    
        confluence_zones = []
        processed = set()
        
        for i, (price1, type1) in enumerate(all_levels):
            if price1 in processed:
                continue
                
            zone = [(price1, type1)]
            for price2, type2 in all_levels[i+1:]:
                if abs(price1 - price2) / price1 < threshold:
                    zone.append((price2, type2))
                    processed.add(price2)
                    
            if len(zone) > 1:
                confluence_zones.append(zone)
                
        if not confluence_zones:
            return "Nenhuma zona de confluência identificada"
            
        return "\n".join(
            f"→ {zone[0][0]:.2f}: " + ", ".join(f"{type}" for _, type in zone)
            for zone in confluence_zones
        )

class TemporalFusionTransformer:
    def __init__(self, config: Config):
        self.config = config
        self.static_variables = ['symbol', 'tick_size', 'point_value']
        self.known_variables = ['hour', 'day_of_week', 'month', 'is_session_start', 'is_session_end']
        self.observed_variables = [
            'VOL.', 'PREÇO ABERT.', 'PREÇO MÍN.', 'PREÇO MÁX.', 'ÚLT. PREÇO', 'AJUSTE',
            'sma_20', 'ema_20', 'adx', 'rsi', 'stoch', 'cci',
            'bb_high', 'bb_low', 'atr', 'obv', 'mfi'
        ]
        
    def variable_selection_network(self, inputs):
        """Rede de seleção de variáveis importantes"""
        grn_units = 64
        dropout_rate = 0.2
        
        # Processa variáveis estáticas
        static_ctx = Dense(grn_units, activation='elu')(inputs['static'])
        static_ctx = LayerNormalization()(static_ctx)
        
        # Processa variáveis conhecidas
        known_combined = Dense(grn_units, activation='elu')(inputs['known'])
        known_ctx = LayerNormalization()(known_combined)
        
        # Processa variáveis observadas
        obs_combined = Dense(grn_units, activation='elu')(inputs['observed'])
        obs_ctx = LayerNormalization()(obs_combined)
        
        # Combina todos os contextos
        combined_ctx = Concatenate()([static_ctx, known_ctx, obs_ctx])
        
        # Aplica seleção de variáveis
        selector = Dense(len(self.observed_variables), activation='sigmoid')(combined_ctx)
        return selector
        
    def build_model(self, input_shape: tuple) -> Model:
        """Constrói o modelo TFT completo"""
        # Entradas
        static_inputs = Input(shape=(len(self.static_variables),))
        known_inputs = Input(shape=(self.config.SEQUENCE_LENGTH, len(self.known_variables)))
        observed_inputs = Input(shape=(self.config.SEQUENCE_LENGTH, len(self.observed_variables)))
        
        # Camada de normalização
        observed_normalized = BatchNormalization()(observed_inputs)
        
        # Seleção de variáveis
        inputs_dict = {
            'static': static_inputs,
            'known': known_inputs,
            'observed': observed_normalized
        }
        variable_weights = self.variable_selection_network(inputs_dict)
        
        # Aplica pesos às variáveis
        weighted_inputs = multiply([observed_normalized, variable_weights])
        
        # Processamento temporal LSTM/GRU
        temporal = LSTM(256, return_sequences=True)(weighted_inputs)
        temporal = Dropout(0.2)(temporal)
        temporal = LayerNormalization()(temporal)
        
        # Processamento GRU
        temporal = GRU(256, return_sequences=True)(temporal)
        temporal = LayerNormalization()(temporal)
        
        # Multi-head Self-Attention
        attention = MultiHeadAttention(
            num_heads=4,
            key_dim=64
        )(temporal, temporal)
        attention_out = LayerNormalization()(attention + temporal)
        
        # Interpretador temporal
        interpreter = Dense(128, activation='relu')(attention_out)
        interpreter = Dropout(0.1)(interpreter)
        
        # Saída quantílica para intervalos de confiança
        outputs = Dense(3, activation='linear')(interpreter)  # q10, q50, q90
        
        model = Model(
            inputs=[static_inputs, known_inputs, observed_inputs],
            outputs=outputs
        )
        
        model.compile(
            optimizer=Adam(self.config.LEARNING_RATE),
            loss='huber',
            metrics=['mae', 'mse']
        )
        
        return model

class MarketAnalyzer:
    def __init__(self, base_dir: str, cache_dir: str):
        self.base_dir = base_dir
        self.cache_dir = cache_dir
        self.model = None
        self.scaler = None
        self.data = {}
        self.last_update = None
        self.setup_logging()
        self.config = Config()
        
    def setup_logging(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(os.path.join(self.cache_dir, 'market_analyzer.log')),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        
   
    def validate_data(self, df: pd.DataFrame) -> bool:
        """Valida os dados carregados"""
        required_columns = [
            'DATA', 'VOL.', 'PREÇO ABERT.', 'PREÇO MÍN.', 
            'PREÇO MÁX.', 'ÚLT. PREÇO', 'AJUSTE'
        ]
        
        # Verifica colunas necessárias
        if not all(col in df.columns for col in required_columns):
            self.logger.error("Colunas obrigatórias ausentes no DataFrame")
            return False
            
        # Verifica dados nulos
        if df[required_columns].isnull().any().any():
            self.logger.warning("Existem valores nulos nos dados")
            
        # Verifica ordem cronológica
        if not df['DATA'].is_monotonic_increasing:
            self.logger.error("Dados não estão em ordem cronológica")
            return False
            
        # Verifica valores negativos em colunas numéricas
        price_columns = ['PREÇO ABERT.', 'PREÇO MÍN.', 'PREÇO MÁX.', 'ÚLT. PREÇO', 'AJUSTE']
        if (df[price_columns] < 0).any().any():
            self.logger.error("Existem preços negativos nos dados")
            return False
            
        return True
    
    def load_data(self) -> bool:
        """Carrega dados apenas se necessário (verificando última modificação)"""
        try:
            latest_file_time = max(
                os.path.getmtime(os.path.join(self.base_dir, f))
                for f in os.listdir(self.base_dir)
                if f.endswith('.csv')
            )
            
            if self.last_update and latest_file_time <= self.last_update:
                return False  # Dados já estão atualizados
                
            self.data = {'WDO': None}
            
            for filename in os.listdir(self.base_dir):
                if filename.endswith(".csv"):
                    ticker = filename.split('_')[0].upper()
                    
                    if ticker in self.data:
                        try:
                            df = pd.read_csv(os.path.join(self.base_dir, filename), encoding="utf-8")
                            if 'DATA' in df.columns:
                                df['DATA'] = pd.to_datetime(df['DATA'])
                            
                            # Adiciona a validação
                            if self.validate_data(df):
                                self.data[ticker] = df
                                self.logger.info(f"Arquivo {ticker} carregado com sucesso. Shape: {df.shape}")
                            else:
                                self.logger.error(f"Dados inválidos para {ticker}")
                                return False
                                
                        except Exception as e:
                            self.logger.error(f"Erro ao carregar {filename}: {str(e)}")
                            continue
            
            self.last_update = latest_file_time
            return True
            
        except Exception as e:
            self.logger.error(f"Erro ao carregar dados: {str(e)}")
            return False
    
        
    def add_technical_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adiciona indicadores técnicos ao DataFrame"""
        df = df.copy()
        
        # Primeiro, verifica se as colunas necessárias existem
        required_cols = ['ÚLT. PREÇO', 'PREÇO MÁX.', 'PREÇO MÍN.', 'VOL.']
        if not all(col in df.columns for col in required_cols):
            self.logger.error(f"Colunas necessárias ausentes: {required_cols}")
            return df
            
        try:
            # Calcula indicadores com tratamento de erro
            df['sma_20'] = ta.trend.sma_indicator(df['ÚLT. PREÇO'], window=self.config.SMA_PERIOD)
            df['ema_20'] = ta.trend.ema_indicator(df['ÚLT. PREÇO'], window=self.config.EMA_PERIOD)
            df['adx'] = ta.trend.adx(df['PREÇO MÁX.'], df['PREÇO MÍN.'], df['ÚLT. PREÇO'], 
                                    window=self.config.ADX_PERIOD)
            df['rsi'] = ta.momentum.rsi(df['ÚLT. PREÇO'], window=self.config.RSI_PERIOD)
            df['stoch'] = ta.momentum.stoch(df['PREÇO MÁX.'], df['PREÇO MÍN.'], df['ÚLT. PREÇO'], 
                                          window=14)
            df['cci'] = ta.trend.cci(df['PREÇO MÁX.'], df['PREÇO MÍN.'], df['ÚLT. PREÇO'], 
                                    window=20)
            df['bb_high'] = ta.volatility.bollinger_hband(df['ÚLT. PREÇO'], window=20)
            df['bb_low'] = ta.volatility.bollinger_lband(df['ÚLT. PREÇO'], window=20)
            df['atr'] = ta.volatility.average_true_range(df['PREÇO MÁX.'], df['PREÇO MÍN.'], 
                                                        df['ÚLT. PREÇO'], window=14)
            df['obv'] = ta.volume.on_balance_volume(df['ÚLT. PREÇO'], df['VOL.'])
            df['mfi'] = ta.volume.money_flow_index(df['PREÇO MÁX.'], df['PREÇO MÍN.'], 
                                                 df['ÚLT. PREÇO'], df['VOL.'], window=14)
            
            # Preenche valores NaN
            df = df.fillna(method='ffill').fillna(method='bfill')
            
            self.logger.info("Indicadores técnicos calculados com sucesso")
            return df
            
        except Exception as e:
            self.logger.error(f"Erro ao calcular indicadores técnicos: {str(e)}")
            return df
        
    def identify_levels(self, df: pd.DataFrame, last_close: float) -> Tuple[List[dict], List[dict]]:
        """Identifica e analisa níveis de suporte e resistência"""
        supports = []
        resistances = []
        
        support_df = df[(df['PREÇO MÍN.'] < last_close) & (df['VOL.'] > MIN_VOLUME)]\
                       .sort_values(by='PREÇO MÍN.', ascending=False)\
                       .head(30)
        
        for _, row in support_df.iterrows():
            support = {
                "price": row['PREÇO MÍN.'],
                "dist": abs((last_close - row['PREÇO MÍN.']) / last_close) * 100,
                "volume": row['VOL.'],
                "date": row['DATA'].strftime("%Y-%m-%d") if 'DATA' in row else 'N/A'
            }
            supports.append(support)
        
        resistance_df = df[(df['PREÇO MÁX.'] > last_close) & (df['VOL.'] > MIN_VOLUME)]\
                          .sort_values(by='PREÇO MÁX.')\
                          .head(30)
        
        for _, row in resistance_df.iterrows():
            resistance = {
                "price": row['PREÇO MÁX.'],
                "dist": abs((row['PREÇO MÁX.'] - last_close) / last_close) * 100,
                "volume": row['VOL.'],
                "date": row['DATA'].strftime("%Y-%m-%d") if 'DATA' in row else 'N/A'
            }
            resistances.append(resistance)
            
        return supports, resistances
        
    def save_report(self, supports: List[dict], resistances: List[dict], last_close: float):
        """Salva o relatório em um arquivo"""
        report_path = os.path.join(self.cache_dir, self.config.REPORT_FILE)
        try:
            # Verifica se todos os indicadores necessários estão presentes
            required_indicators = ['sma_20', 'ema_20', 'adx', 'rsi', 'stoch', 'cci',
                                 'bb_high', 'bb_low', 'atr', 'obv', 'mfi']
            
            if not all(indicator in self.data['WDO'].columns for indicator in required_indicators):
                self.logger.warning("Recalculando indicadores técnicos...")
                self.data['WDO'] = self.add_technical_indicators(self.data['WDO'])
            
            report_generator = MarketReport(self.config)
            report_content = report_generator.generate_report(
                df=self.data['WDO'],
                supports=supports,
                resistances=resistances,
                last_close=last_close
            )
            
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write(report_content)
                
            self.logger.info(f"Relatório salvo em: {report_path}")
            
        except Exception as e:
            self.logger.error(f"Erro ao salvar relatório: {str(e)}")
            raise  
            
class ModelManager:
    def __init__(self, cache_dir: str, seq_len: int = 60):
        self.cache_dir = cache_dir
        self.seq_len = seq_len
        self.feature_columns = [
            'VOL.', 'PREÇO ABERT.', 'PREÇO MÍN.', 'PREÇO MÁX.', 'ÚLT. PREÇO', 'AJUSTE',
            'sma_20', 'ema_20', 'adx', 'rsi', 'stoch', 'cci',
            'bb_high', 'bb_low', 'atr', 'obv', 'mfi'
        ]
     
    def load_model_and_scaler(self):
        """Carrega modelo e scaler existentes"""
        model_path = os.path.join(self.cache_dir, 'model.keras')
        scaler_path = os.path.join(self.cache_dir, 'scaler.pkl')
        
        if os.path.exists(model_path) and os.path.exists(scaler_path):
            try:
                model = load_model(model_path)
                scaler = joblib.load(scaler_path)
                return model, scaler
            except Exception as e:
                logging.error(f"Erro ao carregar modelo/scaler: {str(e)}")
                return None, None
        return None, None
    
    def save_model_and_scaler(self, model: Model, scaler: RobustScaler):
        """Salva modelo e scaler"""
        try:
            model_path = os.path.join(self.cache_dir, 'model.keras')
            scaler_path = os.path.join(self.cache_dir, 'scaler.pkl')
            
            model.save(model_path)
            joblib.dump(scaler, scaler_path)
            return True
        except Exception as e:
            logging.error(f"Erro ao salvar modelo/scaler: {str(e)}")
            return False

    def build_model(self, input_shape: tuple) -> Model:
        """Constrói o modelo com arquitetura otimizada"""
        inputs = Input(shape=input_shape)
        x = BatchNormalization()(inputs)
        
        # LSTM Layer
        lstm = LSTM(256, return_sequences=True)(x)
        lstm = Dropout(0.2)(lstm)
        lstm_out = LayerNormalization()(lstm)
        
        # GRU Layer
        gru = GRU(256, return_sequences=True)(lstm_out)
        gru_out = LayerNormalization()(gru)
        
        # Multi-head Attention
        attention = MultiHeadAttention(num_heads=4, key_dim=64)(gru_out, gru_out)
        attention_out = LayerNormalization()(attention + gru_out)
        
        # Feed-forward network
        ff = Dense(256, activation='relu')(attention_out)
        ff = Dropout(0.2)(ff)
        
        gap = GlobalAveragePooling1D()(ff)
        dense = Dense(128, activation='relu')(gap)
        dense = Dropout(0.1)(dense)
        outputs = Dense(2, activation='linear')(dense)
        
        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=Adam(1e-4), loss='huber', metrics=['mae', 'mse'])
        
        return model
        
    def prepare_data(self, df, scaler):
        """Prepara os dados para o modelo"""
        df_features = df[self.feature_columns].copy()
        data_scaled = scaler.transform(df_features)
        
        X, y = [], []
        for i in range(len(data_scaled) - self.seq_len):
            X.append(data_scaled[i:i+self.seq_len])
            price_idx = self.feature_columns.index('ÚLT. PREÇO')
            adjust_idx = self.feature_columns.index('AJUSTE')
            y.append(data_scaled[i+self.seq_len, [price_idx, adjust_idx]])
            
        return np.array(X), np.array(y)

class EnhancedLevelAnalyzer:
    def __init__(self, config: Config):
        self.config = config
        self.volume_threshold = config.MIN_VOLUME
        
    def calculate_level_strength(self, price: float, volume: float, touches: int,
                               price_action: float, gap: float) -> float:
        """Calcula a força do nível baseado em múltiplos fatores"""
        # Normaliza o volume (0-1)
        volume_score = min(volume / (self.volume_threshold * 2), 1.0)
        
        # Número de toques (mais toques = nível mais forte)
        touch_score = min(touches / 5, 1.0)
        
        # Força do price action (reversões no nível)
        price_action_score = min(price_action / 0.5, 1.0)
        
        # Gap score (distância de outros níveis)
        gap_score = min(gap / 0.2, 1.0)
        
        # Pesos dos componentes
        weights = {
            'volume': 0.4,      # Volume é o fator mais importante
            'touches': 0.25,    # Número de toques é segundo mais importante
            'price_action': 0.2,
            'gap': 0.15
        }
        
        # Calcula score final
        strength = (
            volume_score * weights['volume'] +
            touch_score * weights['touches'] +
            price_action_score * weights['price_action'] +
            gap_score * weights['gap']
        )
        
        return strength
        
    def identify_levels(self, df: pd.DataFrame, last_close: float) -> Tuple[List[dict], List[dict]]:
        """Identifica e analisa níveis de suporte e resistência com análise aprimorada"""
        supports = []
        resistances = []
        
        # Encontra níveis iniciais baseados em volume
        support_df = df[(df['PREÇO MÍN.'] < last_close) & (df['VOL.'] > self.volume_threshold)]
        resistance_df = df[(df['PREÇO MÁX.'] > last_close) & (df['VOL.'] > self.volume_threshold)]
        
        # Processa suportes
        for price_level in support_df['PREÇO MÍN.'].unique():
            level_data = support_df[support_df['PREÇO MÍN.'] == price_level]
            
            # Calcula métricas avançadas
            total_volume = level_data['VOL.'].sum()
            touches = len(level_data)
            price_action = self._calculate_price_action(df, price_level, 'support')
            gap = self._calculate_level_gap(df, price_level, 'support')
            
            strength = self.calculate_level_strength(
                price_level, total_volume, touches, price_action, gap
            )
            
            support = {
                "price": price_level,
                "volume": total_volume,
                "touches": touches,
                "strength": strength,
                "dist": abs((last_close - price_level) / last_close) * 100,
                "dates": level_data['DATA'].dt.strftime("%Y-%m-%d").tolist(),
                "last_test": level_data['DATA'].max().strftime("%Y-%m-%d"),
                "gap": gap
            }
            supports.append(support)
        
        # Processa resistências
        for price_level in resistance_df['PREÇO MÁX.'].unique():
            level_data = resistance_df[resistance_df['PREÇO MÁX.'] == price_level]
            
            # Calcula métricas avançadas
            total_volume = level_data['VOL.'].sum()
            touches = len(level_data)
            price_action = self._calculate_price_action(df, price_level, 'resistance')
            gap = self._calculate_level_gap(df, price_level, 'resistance')
            
            strength = self.calculate_level_strength(
                price_level, total_volume, touches, price_action, gap
            )
            
            resistance = {
                "price": price_level,
                "volume": total_volume,
                "touches": touches,
                "strength": strength,
                "dist": abs((price_level - last_close) / last_close) * 100,
                "dates": level_data['DATA'].dt.strftime("%Y-%m-%d").tolist(),
                "last_test": level_data['DATA'].max().strftime("%Y-%m-%d"),
                "gap": gap
            }
            resistances.append(resistance)
            
        # Ordena por volume e força
        supports.sort(key=lambda x: (x['volume'], x['strength']), reverse=True)
        resistances.sort(key=lambda x: (x['volume'], x['strength']), reverse=True)
        
        return supports[:self.config.TOP_LEVELS], resistances[:self.config.TOP_LEVELS]
        
    def _calculate_price_action(self, df: pd.DataFrame, level: float, 
                              level_type: str, threshold: float = 0.001) -> float:
        """Calcula a força do price action em um nível"""
        if level_type == 'support':
            # Encontra reversões de baixa para alta
            reversals = df[
                (df['PREÇO MÍN.'].between(level * (1-threshold), level * (1+threshold))) &
                (df['ÚLT. PREÇO'] > df['PREÇO ABERT.'])
            ]
        else:
            # Encontra reversões de alta para baixa
            reversals = df[
                (df['PREÇO MÁX.'].between(level * (1-threshold), level * (1+threshold))) &
                (df['ÚLT. PREÇO'] < df['PREÇO ABERT.'])
            ]
            
        return len(reversals) / len(df)
        
    def _calculate_level_gap(self, df: pd.DataFrame, level: float, 
                           level_type: str) -> float:
        """Calcula a distância média para os níveis mais próximos"""
        if level_type == 'support':
            levels_below = df[df['PREÇO MÍN.'] < level]['PREÇO MÍN.'].unique()
            if len(levels_below) > 0:
                return min(abs(level - x) / level for x in levels_below)
        else:
            levels_above = df[df['PREÇO MÁX.'] > level]['PREÇO MÁX.'].unique()
            if len(levels_above) > 0:
                return min(abs(x - level) / level for x in levels_above)
        return 1.0  # Retorna 1 se não houver níveis próximos

def main():
    # Configurações
    config = Config()
    
    # Certifica que o diretório de cache existe
    os.makedirs(config.CACHE_DIR, exist_ok=True)
    
    # Inicializa o analisador e o analisador de níveis
    analyzer = MarketAnalyzer(config.BASE_DIR, config.CACHE_DIR)
    level_analyzer = EnhancedLevelAnalyzer(config)
    
    try:
        # Verifica se precisa atualizar os dados
        if analyzer.load_data():
            analyzer.logger.info("Novos dados carregados. Atualizando análise...")
            
            if analyzer.data['WDO'] is not None:
                # Adiciona indicadores técnicos
                df_with_indicators = analyzer.add_technical_indicators(analyzer.data['WDO'])
                
                # Obtém último fechamento
                last_close = df_with_indicators['ÚLT. PREÇO'].iloc[-1]
                
                # Identifica níveis usando o analisador aprimorado
                supports, resistances = level_analyzer.identify_levels(
                    df_with_indicators, last_close
                )
                
                # Salva relatório
                analyzer.save_report(supports, resistances, last_close)
                
                # Lê e exibe o relatório
                report_path = os.path.join(config.CACHE_DIR, config.REPORT_FILE)
                if os.path.exists(report_path):
                    with open(report_path, 'r', encoding='utf-8') as f:
                        print(f.read())
                    
            else:
                analyzer.logger.error("Arquivo WDO não encontrado no diretório.")
        else:
            analyzer.logger.info("Dados já estão atualizados. Usando relatório existente...")
            # Lê e exibe o relatório existente
            report_path = os.path.join(config.CACHE_DIR, config.REPORT_FILE)
            if os.path.exists(report_path):
                with open(report_path, 'r', encoding='utf-8') as f:
                    print(f.read())
            
    except Exception as e:
        analyzer.logger.error(f"Erro durante a execução: {str(e)}")

if __name__ == "__main__":
    main()

2024-11-12 21:12:56,825 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,833 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,840 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,848 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,854 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,861 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,868 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,876 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,883 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,889 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,897 - INFO - Arquivo WDO carregado com sucesso. Shape: (7, 25)
2024-11-12 21:12:56,903 - INFO - Arquivo WDO carregado com sucesso. Shape: (5, 25)
2024


ANÁLISE DE MERCADO - WDO
Data/Hora: 2024-11-12 21:12:57
Último Fechamento: 5758.00

RESUMO DO MERCADO
Tendência: LATERAL →
Volatilidade: 0.08%
Volume Médio (5 períodos): 867
RSI Atual: 28.3

TOP NÍVEIS DE RESISTÊNCIA (Ordenados por Volume)
 1. Preço:   5,792.00 | Volume:      9,054 [▉▉░░░] | Força: [■■■■■□□□□□] | Toques:  2 | Dist:   0.59% | Último Teste: 2024-11-12 | Gap:  0.0%
    ├─ Histórico: 2024-11-12 → 2024-11-12
 2. Preço:   5,776.00 | Volume:      6,713 [▉▉░░░] | Força: [■■■■■■□□□□] | Toques:  3 | Dist:   0.31% | Último Teste: 2024-11-12 | Gap:  0.0%
    ├─ Histórico: 2024-11-12 → 2024-11-12 → 2024-11-12
 3. Preço:   5,787.50 | Volume:      6,672 [▉▉░░░] | Força: [■■■■■■□□□□] | Toques:  3 | Dist:   0.51% | Último Teste: 2024-11-12 | Gap:  0.0%
    ├─ Histórico: 2024-11-12 → 2024-11-12 → 2024-11-12
 4. Preço:   5,806.00 | Volume:      6,325 [▉▉░░░] | Força: [■■■■■□□□□□] | Toques:  2 | Dist:   0.83% | Último Teste: 2024-11-12 | Gap:  0.0%
    ├─ Histórico: 2024-11-12 → 2024-11-